# Phase 4 — does the "base beats fine-tuned" result hold up?

On AyaTEC the un-fine-tuned GATE-AraBert-v1 beat both fine-tuned models by
about 2x. That contradicts three phases of work, so it needs a second,
independent benchmark before anyone acts on it.

**QRCD** provides one: 169 questions, gold verses recovered from the `pq_id`
ranges, and only **1** question shared with AyaTEC.

Also adds `hybrid RRF (base + BM25)`, which the last run never tried — it
fused BM25 with the *weaker* model.

**T4 GPU → Run all.** ~6 minutes.

### 1. Setup

In [ ]:
# ---- Cell 1: setup ----
import os, sys, shutil, subprocess, json, time, math, re
import numpy as np, statistics as st
import torch
from google.colab import drive
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available(): raise SystemExit("Enable the T4 GPU")
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")

ROOT="/content/drive/MyDrive"
P1=f"{ROOT}/Phase1_Project/MemberB_B4_B6_output"
P1FIX=f"{ROOT}/Phase1_Project/data_fix_output"
GV2=f"{ROOT}/Phase3_Project/guardrail_output_v2"
ROMA=f"{ROOT}/Phase2_Project/Roma_output"
OUT=f"{ROOT}/Phase4_Project"; os.makedirs(f"{OUT}/data", exist_ok=True)
ART="/content/artifacts"; PROJECT="/content/QuranicRAG"
os.makedirs(ART, exist_ok=True); os.makedirs(f"{PROJECT}/quranNLP/shared/data", exist_ok=True)
os.makedirs(f"{PROJECT}/src", exist_ok=True); os.chdir(PROJECT)

shutil.rmtree("/content/_repo", ignore_errors=True)
subprocess.run(["git","clone","--depth","1",
 "https://github.com/Laiba-Noor/quranic-rag-hallucination-free.git","/content/_repo"],check=True)
CSV=f"{PROJECT}/quranNLP/shared/data/final_cross_reference_index.csv"
if not os.path.exists(CSV):
    shutil.copy(f"{P1FIX}/shared_data/final_cross_reference_index.csv", CSV)
for local, remote in {"m_v1": f"{P1}/b5_real_finetuned",
                      "m_v2": f"{GV2}/b5_real_finetuned_v2"}.items():
    if not (os.path.isdir(f"{ART}/{local}") and os.listdir(f"{ART}/{local}")):
        print("copying", local); shutil.copytree(remote, f"{ART}/{local}", dirs_exist_ok=True)
try:
    import hnswlib, sentence_transformers, scipy, rank_bm25   # noqa
    print("deps present")
except ImportError:
    !pip install -q sentence-transformers hnswlib scipy rank_bm25
print("SETUP OK")

### 2. Corpus and both benchmarks

In [ ]:
# ---- Cell 2: corpus + BOTH benchmarks ----
import csv; csv.field_size_limit(sys.maxsize)
rows=list(csv.DictReader(open(CSV, encoding="utf-8")))
VERSES=[(r["verse_key"], (r.get("clean_verse") or "").strip())
        for r in rows if (r.get("clean_verse") or "").strip()]
VALID={k for k,_ in VERSES}
print(f"{len(VERSES)} verses in corpus")

def fetch(name, *cands):
    dst=f"{OUT}/data/{name}"
    if os.path.exists(dst): return dst
    for c in cands:
        if os.path.exists(c): shutil.copy(c, dst); return dst
    from google.colab import files
    print(f"Upload {name}"); up=files.upload()
    shutil.copy(list(up.keys())[0], dst); return dst

# --- benchmark A: AyaTEC (174 answerable questions) ---
aya=json.load(open(fetch("ayatec_records.json",
    "/content/_repo/Data/ayatec_records.json",
    f"{ROMA}/data/ayatec_records.json"), encoding="utf-8"))
BENCH_AYA=[(r["question"], {v for v in r["verse_keys"] if v in VALID})
           for r in aya if r.get("question") and r.get("verse_keys")]
BENCH_AYA=[(q,g) for q,g in BENCH_AYA if g]

# --- benchmark B: QRCD (independent; gold recovered from pq_id ranges) ---
qrcd=json.load(open(fetch("qrcd_flat.json",
    f"{ROMA}/data/qrcd_flat.json",
    "/content/_repo/Data/qrcd_flat.json"), encoding="utf-8"))

def keys_from(pq):
    m=re.match(r"^(\d+):(\d+)(?:-(\d+))?", pq.split("\t")[0].strip())
    if not m: return []
    s,a1,a2=int(m.group(1)), int(m.group(2)), int(m.group(3) or m.group(2))
    return [f"{s}:{a}" for a in range(a1, a2+1)]

agg={}
for r in qrcd:
    q=r.get("question")
    if not q: continue
    agg.setdefault(q,set()).update(k for k in keys_from(r["pq_id"]) if k in VALID)
BENCH_QRCD=[(q,g) for q,g in agg.items() if g]

for nm,b in [("AyaTEC",BENCH_AYA), ("QRCD",BENCH_QRCD)]:
    sz=sorted(len(g) for _,g in b)
    ceil=sum(min(10,len(g))/len(g) for _,g in b)/len(b)
    print(f"{nm:<8} {len(b):>4} questions | gold/q median {sz[len(sz)//2]:>3} "
          f"max {sz[-1]:>3} | ceiling R@10 = {ceil:.4f}")

# sanity: the two benchmarks must not be the same questions
qa={q for q,_ in BENCH_AYA}; qq={q for q,_ in BENCH_QRCD}
print(f"overlap between benchmarks: {len(qa & qq)} questions")

### 3. Metrics

In [ ]:
# ---- Cell 3: metrics ----
def metrics_from_ranked(ranked, gold, ks=(1,5,10,20)):
    out={"MRR": next((1.0/i for i,vk in enumerate(ranked,1) if vk in gold), 0.0)}
    dcg=sum(1/math.log2(i+1) for i,vk in enumerate(ranked[:10],1) if vk in gold)
    idcg=sum(1/math.log2(i+1) for i in range(1,min(len(gold),10)+1))
    out["NDCG@10"]=dcg/idcg if idcg else 0.0
    for k in ks:
        h=sum(1 for vk in ranked[:k] if vk in gold)
        out[f"Recall@{k}"]=h/len(gold); out[f"HitRate@{k}"]=1.0 if h else 0.0
    return out

def evaluate_ranker(rank_fn, bench, want=30):
    from collections import defaultdict
    acc=defaultdict(list); per={"r10":[], "hit20":[], "rr":[]}
    for q,gold in bench:
        m=metrics_from_ranked(rank_fn(q, want), gold)
        for k,v in m.items(): acc[k].append(v)
        per["rr"].append(m["MRR"]); per["r10"].append(m["Recall@10"])
        per["hit20"].append(m["HitRate@20"])
    return {k: sum(v)/len(v) for k,v in acc.items()}, per

def show(title, rows):
    print(f"\n{title}")
    print(f"{'system':<36}{'R@10':>8}{'R@20':>8}{'Hit@10':>9}{'Hit@20':>9}{'MRR':>8}{'NDCG':>8}")
    print("-"*86)
    for n,r in rows:
        print(f"{n:<36}{r['Recall@10']:>8.4f}{r['Recall@20']:>8.4f}"
              f"{r['HitRate@10']:>9.4f}{r['HitRate@20']:>9.4f}{r['MRR']:>8.4f}{r['NDCG@10']:>8.4f}")

def compare(label, a, b, keys=("r10","hit20")):
    from scipy.stats import wilcoxon
    for kk in keys:
        x,y=a[kk],b[kk]; d=[q-p for p,q in zip(x,y)]
        if not any(d): print(f"{label} {kk:<6} identical"); continue
        _,p=wilcoxon(x,y,zero_method="wilcox"); sd=st.pstdev(d) or 1e-9
        print(f"{label} {kk:<6} delta={st.mean(d):+.4f} p={p:.4g} "
              f"d={st.mean(d)/sd:+.3f} {'SIGNIFICANT' if p<0.05 else 'ns'}")
print("metrics ready")

### 4. Retrievers

In [ ]:
# ---- Cell 4: build all retrievers ----
from sentence_transformers import SentenceTransformer
import hnswlib
from rank_bm25 import BM25Okapi

KEYS=[k for k,_ in VERSES]

def build_dense(path, tag):
    m=SentenceTransformer(path); m.max_seq_length=64
    emb=m.encode([t for _,t in VERSES], convert_to_numpy=True, normalize_embeddings=True,
                 show_progress_bar=False, batch_size=256).astype(np.float32)
    ix=hnswlib.Index(space="cosine", dim=emb.shape[1])
    ix.init_index(max_elements=len(VERSES), ef_construction=200, M=16)
    ix.add_items(emb, ids=np.arange(len(VERSES)))
    def rank(q, want=30):
        k=min(max(want*2,64), len(VERSES)-1)
        while k>=1:
            try:
                ix.set_ef(min(max(k*2,64), len(VERSES)))
                lab,_=ix.knn_query(m.encode([q], convert_to_numpy=True,
                                            normalize_embeddings=True), k=k)
                break
            except RuntimeError: k//=2
        else: return []
        seen,out=set(),[]
        for i in lab[0]:
            vk=KEYS[i]
            if vk not in seen: seen.add(vk); out.append(vk)
            if len(out)>=want: break
        return out
    print("  built", tag)
    return rank

AR_DIAC=re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED\u0640]")
def norm(t):
    t=AR_DIAC.sub("", t)
    t=re.sub(r"[إأآٱ]","ا",t); t=re.sub(r"ى","ي",t); t=re.sub(r"ة","ه",t)
    return re.sub(r"[^\u0600-\u06FF\s]"," ",t).split()

bm25=BM25Okapi([norm(t) for _,t in VERSES])
def bm25_rank(q, want=30):
    order=np.argsort(-bm25.get_scores(norm(q)))[:want*3]
    seen,out=set(),[]
    for i in order:
        vk=KEYS[i]
        if vk not in seen: seen.add(vk); out.append(vk)
        if len(out)>=want: break
    return out

def rrf(rankers, k=60):
    def rank(q, want=30):
        sc={}
        for r in rankers:
            for i,vk in enumerate(r(q, want*2),1): sc[vk]=sc.get(vk,0.0)+1.0/(k+i)
        return [vk for vk,_ in sorted(sc.items(), key=lambda x:-x[1])][:want]
    return rank

t=time.time()
R={}
R["base GATE-AraBert-v1 (no fine-tune)"]=build_dense(
    "Omartificial-Intelligence-Space/GATE-AraBert-v1","base")
R["v1 fine-tuned"]=build_dense(f"{ART}/m_v1","v1")
R["v2 fine-tuned (+AyaTEC)"]=build_dense(f"{ART}/m_v2","v2")
R["BM25 lexical"]=bm25_rank
R["hybrid RRF (base + BM25)"]=rrf([R["base GATE-AraBert-v1 (no fine-tune)"], bm25_rank])
R["hybrid RRF (v2 + BM25)"]=rrf([R["v2 fine-tuned (+AyaTEC)"], bm25_rank])
print(f"all retrievers ready in {(time.time()-t)/60:.1f} min")

### 5. Results

In [ ]:
# ---- Cell 5: evaluate on BOTH benchmarks ----
ORDER=["base GATE-AraBert-v1 (no fine-tune)","v1 fine-tuned","v2 fine-tuned (+AyaTEC)",
       "BM25 lexical","hybrid RRF (base + BM25)","hybrid RRF (v2 + BM25)"]
ALL={}
for bname, bench in [("AyaTEC (174 q)", BENCH_AYA), ("QRCD (independent)", BENCH_QRCD)]:
    res, per = {}, {}
    for n in ORDER:
        res[n], per[n] = evaluate_ranker(R[n], bench)
    ALL[bname]=(res, per)
    ceil=sum(min(10,len(g))/len(g) for _,g in bench)/len(bench)
    show(f"{bname}  —  ceiling R@10 = {ceil:.4f}", [(n,res[n]) for n in ORDER])
    b="base GATE-AraBert-v1 (no fine-tune)"
    print()
    compare("  v2      vs base ", per[b], per["v2 fine-tuned (+AyaTEC)"])
    compare("  hybrid  vs base ", per[b], per["hybrid RRF (base + BM25)"])

json.dump({b:{"results":r} for b,(r,_) in ALL.items()},
          open(f"{OUT}/phase4_two_benchmarks.json","w"), indent=2, ensure_ascii=False)
print("\nsaved:", f"{OUT}/phase4_two_benchmarks.json")